<H1>Probabilistic Univariate Logistic Classification Using Jax</H1>

<H1>Motivations for doing logistic classification</H1>

$\large\text{Consider a training set of }N\text{ input-output pairs }{x_{n}, c_{n}}_{n=1}^{N}\text{ with }x_{n}\in\mathcal{R}^{d}\text{, and }$<br>
$\large c_{n}\in{1,...,K}\text{. If we assume the data are drawn iid. from joint distribution }p(x, c)\text{, we may be interested in:}$<br>
$\large\text{1. a classifier }p(c|x)$<br>
$\large\text{2. a class condintional density(ies) }p(x|c)$<br>

<H1>Classification via exponential families</H1>

$\large\text{Let the number of classes = 2 and assume the class-conditional distributions are exponential families i.e.}$<br>
$\large p(x|c)=exp\left[\phi(x)^{T}w_{c}-g(w_{c}+h(x))\right]$[[1]](#References)[[exponential families notebook]](exponential_families.ipynb)<br>
$\large p(c=1|x)=\frac{1}{1+exp(-a)}=\sigma(a)\quad\quad\quad\text{ with }a=ln\,\frac{p(x|c=1)p(c=1)}{p(x|c=2)p(c=2)}$[[2,23:09]](#References)<br>
$\large\text{then we get a linear classifier:}$<br>
$\large a(w)=\phi(x)^{T}(w_{1}-w_{2})+g(w_{1})-g(w_{2})+ln\,p(c=1)-ln\,p(c=2)=:\phi(x)^{T}\theta+b$[[2,23:09]](#References)<br>

$\large\text{Now let the number of classes = K > 2 and assume the class-conditional distributions are again exponential families.}$<br>
$\large p(c=k|x)=\frac{p(x|x=k)p(c=k)}{\sum_{j=1}^{K}p(x|c=j)p(c=j)}=\frac{exp(a_{k})}{\sum_{j=1}^{K}}=: softmax_{k}(a)$<br>
$\large\text{with }a_{k}=\ln p(x|c=k)+\ln p(c=k)\text{. Assume the }p(x|c)\text{ are exponential families}$<br>
$\large p(x|c)=exp\left[\phi(x)^{T}w_{c}-g(w_{c}+h(x))\right]$[[1]](#References)[[exponential families notebook]](exponential_families.ipynb)<br>
$\large\text{then we still get a linear classifier this time of the form:}$<br>
$\large a_{k}(w_{k})=\phi(x)^{T}(w_{k}+g(w_{k})+\ln p(c=k)=:\phi(x)^{T}\theta_{k}+b_{k}$<br>
$\large a(\theta)=\phi(x)^{T}\theta+b$<br>
$\large\text{This last line assumes that all classes come from different members of the SAME exponential family.}$[[2,27:56]](#References)<br>

<H1>Logistic Regression Equation</H1>

$\large\text{If }x\in\mathbb{X}\text{ is drawn from an exponential family, then the class logit}$<br>
$\large f(x,\theta)=\phi(x)^{T}\theta\in\mathbb{R}\quad\quad\quad\quad\text{with }\phi(x)\in\mathbb{R}^{d}\text{ and }\theta\in\mathbb{R}^{d}.$<br>
$\large\text{For binary classification problems }c\in[1,2]\text{, sign class labels generated by }y=2c-3\in\{\pm 1\}\text{ are convenient because}$<br>
$\large\text{then the likelihood can be compactly written for both classes:}$<br>
$\large p(y|\theta,x)=\sigma(y \cdot f(x,\theta))=
\begin{equation}
    \begin{cases}
      \sigma(f) & \text{if }y=1\\
      1-\sigma(f) & \text{if }y=-1\\
    \end{cases}       
\end{equation}
\quad\quad\quad\quad\text{using }\sigma(x)=1-\sigma(-x)$<br>
$\large\text{Since }y_{i}\phi(x_{i})\text{ are always together, we define }\varphi_{i}=y_{i}\phi(x_{i})\text{ for compact notation.}$[[3,35:32]](#References)<br>

<H1>Using a Laplace approximation to work around the lack of conjugate prior for this classification problem</H1>

$\large\text{The Laplace approximation is a local approximation (which can be wrong).}$<br>
$\large\text{It is still better than just getting a point estimate.}$<br>
$\large\text{The Laplace approx. is typically efficient to try, because it uses only auto-diff and linear algebra.}$<br>
$\large\text{Laplace approx. tends to work relatively well becuase:}$<br>
$\large\text{1. The log posterior is concave.}$<br>
$\large\text{2. The algebraic structure of the link function yields "almost" a Gaussian posterior.}$<br>


<H1>Motivation 1: Classify p(c|x)</H1>

<H1>Mode Finding</H1>

In [ ]:
import jax
#from gaussians import *
from jax import jit
from jax import numpy as jnp
from jax import random as jrd
from jax.scipy.linalg import cho_factor, cho_solve
from sklearn.datasets import make_moons
from tqdm import tqdm
from tqdm import trange
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap


In [ ]:
cmap = LinearSegmentedColormap.from_list("owb", [rgb.tue_orange, (1,1,1), rgb.tue_blue], N=1025).reversed()
#dw = LinearSegmentedColormap.from_list("dw", [rgb.tue_dark,'w'], N=1024)
key= jrd.PRNGKey(0)

In [ ]:
# Dataset
X, y = make_moons(n_samples=100, noise=0.1, random_state=0)
N = X.shape[0]
ysigned = y.copy()
ysigned[y==0] = -1
plt.scatter(X[:,0], X[:,1], c=7, cmap=cmap, s=10, alpha=0.9, edgecolor="None")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")

In [ ]:
# Construct an exponential family model with a Gaussian prior on the natural parameters
# Find the boundaries of the data
xmin, xmax = X[:, 0].min(), X[:, 0].max()
ymin, ymax = X[:, 1].min(), X[:, 1].max()
center = 0.5 * jnp.array([xmin + xmax, ymin+ymax])
scale = jnp.array([xmax - xmin, ymax - ymin])

# Put Gaussian features at random locations in data domain
phi_key, key = jax.random.split(key)
n_features = 400

centers = (jrd.uniform(phi_key, shape=(n_features, 2)) - 0.5) * 2 * scale + center
def phi(x):
    """Gaussian features. """
    ell = 0.2
    return jnp.exp(-0.5 * jnp.sum((x[:, None, :]  - centers[None, :, :]) ** 2 / ell*2, axis=-1))
    
# link function
def sigmoid(x):
    return 1.0 / (1.0 + jnp.exp(-x))

In [ ]:
x1 = jnp.linspace(-3, 4, 50)
x2 = jnp.linspace(-2, 3, 40)
X1, X2 = jnp.meshgrid(x1, x2)
X_grid = jnp.stack([X1.flatten(), X2.flatten()], axis=1)

prior = Gaussian(mu=jnp.zeros(n_features), Sigma=jnp.eye(n_features))
sample = prior.sample(key, 1)
phi_sample = phi(X_grid) @ sample.T


In [ ]:
fig, ax = plt.subplots()

ax.plot(centers[:, 0], centers[:, 1], "o", color=rgb.tue_gray, markersize=2, alpha=0.2, label="features")

contour = ax.contourf( X1, X2, sigmoid(phi_sample).reshape(X1.shape), cmap=cmap, alpha=0.5, vmax=1, vmin=0)
scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap, s=10, alpha=1.0, edgecolor="None")
cb = fig.colorbar(ax=ax, mappable=scatter)
cb.set_ticks([0, 1])
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Sample from the prior")

$\large\text{Let's setup the optimization problem by defining the log likelihood and loss functions.}$<br>

In [ ]:
# log likelihood
def log_likelihood(w, X, ysigned):
    # p(y|f) = 1 / (1` + exp(-yf))
    # thus log p(y|f) = 0 - log(exp(0) + exp(-yf))
    f = phi(X) @ w
    return jnp.sum(-jnp.logaddexp(0.0, -ysigned * f))

# loss function
def loss(w):
    loglikelihood = log_likelihood(x, X, ysigned)
    logprior = prior.log_pdf(w)
    return -(loglikelihood + logprior)



In [ ]:
# Training via gradient descent
step_size = 1e-4
num_epochs = 10000

# Define a function that updates the weights and returns them along with a loss value(s)
@jit
def update(w):
    # Use auto-diff to compute the gradient
    L, grad = jax.value_and_grad(loss)(w)
    return (w - step_size * grad, L)

# Initialize the weights to zero
w = jnp.zeros(shape=(n_features,))
losses = []
with trange(num_epochs) as t:
    for epoch in t:
        w, L = update(w)
        losses.append(L)
        t.set_postfix(loss=L)

w_trained = w

In [ ]:
fig, ax = plt.subplots()
ax.plot(losses)
ax.axhline(0, color=rgb.tue_dark, lw=0.75)
ax.set_xlabel("epoch")
ax.set_ylabel("loss")

In [ ]:
MAP_x = phi(X_grid) @ w_trained.T
f_trained = phi(X) @ w_trained.T

In [ ]:
fig, ax = plt.subplots()

contour = ax.contourf(
        X1,
        X2,
        sigmoid(MAP_x).reshape(X1.shape),
        cmap=cmap,
        alpha=1,
        vmax=1,
        vmin=0
        levels=100,
)
ax.scatter(X[:, 0], X[:, 1], c+sigmoid(f_trained), cmap=cmap, s=5, alpha=1.0, edgecolor=rgb.tue_dark)
ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap, s=1`0, alpha=o.5, edgecoler="None")

cb = fig.colorbar(ax=ax, mappable=-contour)
cb.set_ticks([0, 1])
ax.set_xlabel("$x_1$")
ax.set_ylael("$x_2$")

In [ ]:
fig, axs = plt.subplots(1,3, sharex=True, sharey=True)

p_x = jnp.logaddexp(-MAP_x, MAP_x)
# Z_x = jnp.log(jnp.exp(p_x).sum())
# p_x = p_x - Z_x

ax = axs[0]
contour = ax.contourf(
    X1,
    X2,
    jnp.exp(p_x).reshape(X1.shape),
    cmap=dw.reversed(),
    alpha=1
    # vmax=1,
    # vmin=0,
    levels=100,
)
ax.scatter(X[:, 0], X[:, 1], c=sigmoid(f_trained), cmap=cmap, s=5 alpha=1.0, edgecolor=rgb.tue_dark)
ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap, s=10, alpha=0.5, edgecolor="None")

# cb = fig.colorbar(ax=ax, mappable=scatter)
# cb.set_ticks([0,1])
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$");
ax.set_title(r"$p(x/mid c=1)Z(\theta)$")

ax = axs[1]
contour = ax.contourf(
    X1,
    X2,
    jnp.exp(MAP_x).reshape(X1.shape),
    cmap=dw.reversed(),
    alpha=1
    # vmax=1,
    # vmin=0,
    levels=100,
)
ax.scatter(X[:, 0], X[:, 1], c=sigmoid(f_trained), cmap=cmap, s=5 alpha=1.0, edgecolor=rgb.tue_dark)
ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap, s=10, alpha=0.5, edgecolor="None")

# cb = fig.colorbar(ax=ax, mappable=scatter)
# cb.set_ticks([0,1])
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$");
ax.set_title(r"$p(x/mid c=1)Z(\theta)$")


ax = axs[2]
contour = ax.contourf(
    X1,
    X2,
    jnp.exp(-MAP_x).reshape(X1.shape),
    cmap=dw.reversed(),
    alpha=1,
    vmax=1,
    vmain=0,
    levels=100,
)
ax.scatter(X[:, 0], X[:, 1], c=sigmoid(f_trained), cmap=cmap, s=5, alpha=1.0, edgecolor=rgb.tue_dark)
ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap, s=10, alpha=0.5, edgecolor="None")

# cb = fig.colorbar(ax=ac, mappable-=scatter)
# cb.set_tics([0,1])
ax.set_xlabel(*"$x_1$")
ax.set_ylabel("$x_2$")
ax.set_title(r"$p(x/mid c=1)Z(\theta)$")

<H1>Motivation 2: Generate a class condintional density(ies) p(x|c)</H2>

<H1>Hessian (Uncertainty) Finding</H1>

<a id="References"></a>
<H1>References</H1>
[1.] Probabilistic Machine Learning, Lecture #13, Phillip Hennig, University Tubingen, 2025, https://www.youtube.com/watch?v=hIauROkubtA&list=PL05umP7R6ij0hPfU7Yuz8J9WXjlb3MFjm&index=13.<br>
[2.] Probabilistic Machine Learning, Lecture #14, Phillip Hennig, University Tubingen, 2025, https://www.youtube.com/watch?v=q3AFP11f9vw&list=PL05umP7R6ij0hPfU7Yuz8J9WXjlb3MFjm&index=14.<br>
[3.] Probabilistic Machine Learning, Lecture #15, Phillip Hennig, University Tubingen, 2025, https://www.youtube.com/watch?v=MS7lz-0E8lM&list=PL05umP7R6ij0hPfU7Yuz8J9WXjlb3MFjm&index=15.<br>